# Prophet and Chronos

Compare medians, intervals, and held-out errors. Which observations were visible?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/amazon/chronos-t5-mini

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors" "xgboost==2.1.4" "lightgbm==4.6.0" "tabpfn==2.2.1" "prophet==1.5.0" "chronos-forecasting==2.3.2"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import torch
torch.set_num_threads(1)
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import (
    load_breast_cancer, load_iris, load_wine, make_classification
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('🖥️ Environment ready')

In [ ]:
from prophet import Prophet

# Generate 3 years of daily sales: trend + yearly + weekly seasonality + noise
n_days = 3 * 365
dates = pd.date_range('2021-01-01', periods=n_days, freq='D')
t = np.arange(n_days)

trend  = 100 + 0.05 * t
yearly = 15 * np.sin(2 * np.pi * t / 365.25)
weekly = 5  * np.sin(2 * np.pi * t / 7)
noise  = np.random.normal(0, 5, n_days)
y = trend + yearly + weekly + noise

df = pd.DataFrame({'ds': dates, 'y': y})
print(f'📅 Generated {len(df)} days of synthetic sales data')

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
)
model.fit(df)

future = model.make_future_dataframe(periods=180)
forecast = model.predict(future)

fig = model.plot(forecast)
plt.title('Prophet Forecast: Sales (3 years history → 6 months ahead)', fontweight='bold')
plt.tight_layout(); plt.show()

fig2 = model.plot_components(forecast)
plt.tight_layout(); plt.show()

print('💡 Prophet decomposes the series into trend + seasonality components — interpretable by design.')

In [ ]:
import torch
from chronos import ChronosPipeline

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-mini',
    device_map='cuda' if torch.cuda.is_available() else 'cpu',
    torch_dtype=torch.float32,
)
print('✅ Chronos-Mini loaded')

context_len = 800
horizon = 60
context = torch.tensor(df['y'].iloc[:context_len].values, dtype=torch.float32)

forecast = pipeline.predict(
    context,
    prediction_length=horizon,
    num_samples=100,
)
fc = forecast.detach().cpu().numpy().squeeze(0)  # (100 samples, horizon)

low, median, high = np.quantile(fc, [0.1, 0.5, 0.9], axis=0)
fc_dates = df['ds'].iloc[context_len:context_len + horizon].values
actual   = df['y'].iloc[context_len:context_len + horizon].values

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['ds'].iloc[:context_len], df['y'].iloc[:context_len], color='gray', alpha=0.6, label='Context')
ax.plot(fc_dates, actual, '--', color='black', label='Actual')
ax.plot(fc_dates, median, color='#26B7DB', linewidth=2, label='Chronos median')
ax.fill_between(fc_dates, low, high, color='#26B7DB', alpha=0.25, label='80% interval')
ax.set_title('Chronos: Zero-Shot Probabilistic Forecast', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Sales'); ax.legend()
plt.tight_layout(); plt.show()

print('💡 No training on this data — Chronos forecasts directly from the pretrained checkpoint.')

## Demo code

Compare medians, intervals, and held-out errors. Which observations were visible?

In [ ]:
samples = pipeline.predict(context, prediction_length=horizon, num_samples=100)
low, median, high = np.quantile(samples.detach().cpu().numpy()[0], [0.1, 0.5, 0.9], axis=0)
print("Forecast horizon:", len(median))
print("Held-out MAE:", np.abs(median - actual).mean())

In [ ]:
# Fit Prophet on the same context window used for Chronos
df_ctx = df.iloc[:context_len].copy()

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
)
m.fit(df_ctx)
future = m.make_future_dataframe(periods=horizon)
prophet_fc = m.predict(future).iloc[-horizon:]

# Compare via MAE
mae_prophet = np.abs(prophet_fc['yhat'].values - actual).mean()
mae_chronos = np.abs(median - actual).mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, name, fc_vals, color, mae in [
    (axes[0], 'Prophet (fitted)',    prophet_fc['yhat'].values, '#26C281', mae_prophet),
    (axes[1], 'Chronos (zero-shot)', median,                    '#26B7DB', mae_chronos),
]:
    ax.plot(df['ds'].iloc[:context_len], df['y'].iloc[:context_len], color='gray', alpha=0.5, label='Context')
    ax.plot(fc_dates, actual, '--', color='black', label='Actual')
    ax.plot(fc_dates, fc_vals, color=color, linewidth=2, label=name)
    ax.set_title(f'{name}  |  MAE = {mae:.2f}', fontweight='bold')
    ax.legend()
plt.tight_layout(); plt.show()

print(f'\n📊 Prophet MAE: {mae_prophet:.2f}    Chronos MAE: {mae_chronos:.2f}')
print('💡 Prophet trains on this series — interpretable, per-dataset.')
print('   Chronos uses a pretrained model — zero-shot, probabilistic out of the box.')

## Interpretation

Synthetic sales data; the comparison shares the context window and forecast horizon.

Compare the output with the evidence in the input. A generated answer can be wrong.